In [2]:
import joblib
import pandas as pd

# Load saved package
data = joblib.load("../models/saved_models/logistic_model.pkl")

model = data["model"]
encoder = data["encoder"]
feature_columns = data["feature_columns"]

C:\Users\User\anaconda3\envs\torchgpu\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\User\anaconda3\envs\torchgpu\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
df = pd.read_csv("../data/final/labeled_dataset.csv")

X = df[["plant_type", "soil_pct", "temperature", "humidity", "light"]]
y = df["water_needed"]

In [5]:
plant_encoded = encoder.transform(X[["plant_type"]])

plant_encoded_df = pd.DataFrame(
    plant_encoded,
    columns=encoder.get_feature_names_out(["plant_type"])
)

X_final = pd.concat(
    [plant_encoded_df, X.drop("plant_type", axis=1).reset_index(drop=True)],
    axis=1
)

checking whether it is overfitting

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc = accuracy_score(y_test, model.predict(X_test))

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Train Accuracy: 0.9964310159382028
Test Accuracy: 0.9953070003910833


Cross validation

In [7]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X_final, y, cv=5)

print("Cross-validation scores:", scores)
print("Mean CV score:", scores.mean())

Cross-validation scores: [0.98904967 0.99139617 1.         0.90788187 0.98063759]
Mean CV score: 0.9737930590324677


In [ ]:
testing data

In [8]:
sample = pd.DataFrame({
    "plant_type": ["money_plant"],
    "soil_pct": [28],
    "temperature": [30],
    "humidity": [70],
    "light": [20]
})

In [9]:
sample_enc = encoder.transform(sample[["plant_type"]])

sample_enc_df = pd.DataFrame(
    sample_enc,
    columns=encoder.get_feature_names_out(["plant_type"])
)

sample_final = pd.concat(
    [sample_enc_df, sample.drop("plant_type", axis=1)],
    axis=1
)

In [10]:
model.predict(sample_final)
model.predict_proba(sample_final)

array([[0.01153228, 0.98846772]])

In [13]:
def test_sample(plant, soil, temp, humidity, light):
    sample = pd.DataFrame({
        "plant_type": [plant],
        "soil_pct": [soil],
        "temperature": [temp],
        "humidity": [humidity],
        "light": [light]
    })
    
    # encode
    sample_enc = encoder.transform(sample[["plant_type"]])
    
    sample_enc_df = pd.DataFrame(
        sample_enc,
        columns=encoder.get_feature_names_out(["plant_type"])
    )
    
    sample_final = pd.concat(
        [sample_enc_df, sample.drop("plant_type", axis=1)],
        axis=1
    )
    
    # predict
    pred = model.predict(sample_final)[0]
    prob = model.predict_proba(sample_final)[0][1]
    
    print("Prediction:", pred)
    print("Probability (water):", prob)

Money plant dry

In [14]:
test_sample("money_plant", 25, 30, 70, 20)

Prediction: 1
Probability (water): 0.9999933544139937


Money plant wet

In [20]:
test_sample("money_plant", 56, 30, 70, 20)

Prediction: 0
Probability (water): 4.485642876435382e-29


In [27]:
test_sample("cactus", 22, 30, 70, 20)

Prediction: 1
Probability (water): 0.8640367751378708


In [30]:
test_sample("snake_plant", 86, 30, 70, 20)

Prediction: 0
Probability (water): 2.2312691418665152e-67
